# Homework 4: K-NN & Cross-Validation
**Name:** Nicholas Starace  
**Student ID:** 4556822  
**D Value:** 22  

## Question 1: Implementing K-Fold Cross Validation

### Part A: Loading and Understanding the Data

In [84]:
import numpy as np
import scipy.io
D = 22

# Load Second Dataset
data = scipy.io.loadmat('input/hw3_data2.mat')

# Read keys and assign data
Ximport = data['X']
y = data['y']
y = y.ravel()                       # make sure numpy vector, NOT matrix

numFeatures = Ximport.shape[1]
numSamples = y.shape[0]
numClasses = y.max()
classDist = numSamples / numClasses
print(f"Number of Features: {numFeatures}")
print(f"Number of Samples: {numSamples}")
print(f"Number of Classes: {numClasses}")
print(f"Class Distribution: {classDist}")

Number of Features: 4
Number of Samples: 150
Number of Classes: 3
Class Distribution: 50.0


**Text Response:** COME BACK TO

### Part B: Implementing K-Fold Split Function

In [85]:
def create_k_folds(n_samples, k, random_seed):
    # Shuffle Indices Randomly
    seedD = int(random_seed)
    np.random.seed(seedD)
    randArr = np.random.permutation(n_samples)

    # Find samples per fold
    baseSize = n_samples // k
    remainder = n_samples % k
    if remainder > 0:           # Accomodate if bins dont split equally
        maxSize = baseSize + 1
    else:
        maxSize = baseSize

    # Divide samples into fold bins
    allFolds = np.ones((k, maxSize), dtype = int) * (-1) # Keep -1 for bins that aren't same as max size

    start = 0
    for i in range(k):
        foldSize = baseSize # Start with smallest size
        if remainder > i:
            foldSize = foldSize + 1 
        
        # Construct fold
        end = start + foldSize
        allFolds[i, 0:foldSize] = randArr[start:end]
        start = end

    return allFolds

In [86]:
k = 5
allFolds = create_k_folds(numSamples, k, D)

# Print Resulting Fold Bins
for i in range(k):
    print(f"Fold {i+1}: {allFolds[i].shape[0]} samples")

print(f"Total: {allFolds.shape[0] * allFolds.shape[1]} samples")

Fold 1: 30 samples
Fold 2: 30 samples
Fold 3: 30 samples
Fold 4: 30 samples
Fold 5: 30 samples
Total: 150 samples


**Text Output:** Each fold bin has 30 samples

### Part C: Implementing Cross-Validation Evaluation

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
def cross_validate_knn(X, y, k_neighbors, k_folds, fold_indices):

    # Separate data in testing and training folds
    for i in range(k_folds):
        testIdx = fold_indices[i, :]      # Selected index range for testing
        testIdx = testIdx[testIdx != -1]  # Remove -1 that was padded

        trainIdxLeft = fold_indices[0:i, :].flatten() # Get all rows before test row
        trainIdxRight = fold_indices[i+1:k_folds, :].flatten() # Get all rows after test row
        trainIdx = np.concatenate((trainIdxLeft, trainIdxRight)) # Create matrix of all rows not including test
        trainIdx = trainIdx[trainIdx != -1] # Remove -1 padded

        X_test = X[testIdx]        # X_test formed from selected fold index 
        X_train = X[trainIdx]      # X_train formed from selected fold indices
        y_test = y[testIdx]
        y_train = y[trainIdx]

        # Train KNN (aquired from SKLearn site)
        KNN = KNeighborsClassifier(n_neighbors=k_neighbors)
        KNN.fit(X_train, y_train)

        # Predict X_train for each iteration
        KNNPredict = KNN.predict(X_test)
    

    

In [88]:
# Run on hw3_data2
k_neighbors = 3 # Number of neighbors

# Ximport, y from mat file, k from separating earlier, allFolds from create_k_folds function
cross_validate_knn(Ximport, y, k_neighbors, k, allFolds)